# The Colorado River: A System Under Strain

Seven states and Mexico share water that starts as snow in the Rocky Mountains,
travels a thousand miles through two of the country's largest reservoirs, and
has been promised out faster than the river delivers it. This notebook tells
that story in the river's own data: the snowpack that feeds it, the runoff it
produces, the timing of its flow, and the reservoirs that used to be able to
absorb a bad year but increasingly can't.

In [ ]:
from __future__ import annotations

import logging

from IPython.display import Markdown, display

from colorado_river_viz.cache import load_snotel_stations, refresh_story
from colorado_river_viz.charts.ch2_snow import build_snow_spaghetti
from colorado_river_viz.charts.ch3_runoff import (
    build_efficiency_trend,
    build_snow_vs_runoff,
    residual_efficiency_trend,
)
from colorado_river_viz.metrics.trend import theil_sen_trend
from colorado_river_viz.narrative import (
    describe_peak_swe,
    describe_runoff_year,
    describe_trend,
)
from colorado_river_viz.settings import Settings
from colorado_river_viz.story_tables import (
    runoff_vs_snow,
    snow_annual,
    snow_index_daily,
    snow_index_envelope,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

REFRESH = "offline"  # "incremental" or "full" to hit the live APIs instead

settings = Settings()
refresh_story(REFRESH, settings)
stations = load_snotel_stations(settings.cache_dir)

## Meet the river

*(map -- T24)*

## 1 · Promised more than it has

*(chapter 1 -- T19)*

## 2 · It starts as snow

Every drop of Colorado River water begins as snow in the mountains of Colorado,
Wyoming, Utah, and New Mexico. A fixed set of 54 long-record SNOTEL stations
(decision 0013) tracks the basin's snow water equivalent (SWE) -- the depth of
water the snowpack would produce if it melted all at once -- every winter back
to the early 1980s.

In [ ]:
snow_daily = snow_index_daily(settings.cache_dir, stations)
snow_envelope = snow_index_envelope(snow_daily)
snow_year = snow_annual(settings.cache_dir, stations, snow_daily)

fig_snow = build_snow_spaghetti(snow_daily, snow_envelope)
fig_snow.show()

In [ ]:
latest_snow = snow_year.dropna(subset=["peak_swe_in"]).iloc[-1]
display(Markdown(f"**{describe_peak_swe(latest_snow)}**"))

**How we know:** Basin SWE is the mean across the 54 index stations reporting
that day, requiring at least 90% of the index to report (decision 0017); gaps
of 7 days or less are linearly interpolated first. `pct_of_median` compares the
sum of reporting stations' SWE against the sum of their own 1991-2020 NRCS
medians -- the standard basin-index method. Source: NRCS SNOTEL, `AWDB` API.

## 3 · Same snow, less river

A given amount of mountain snowpack no longer turns into the same amount of
spring runoff it once did. Warmer soils and thirstier vegetation intercept more
of the melt before it reaches a gauge, so the relationship between peak SWE and
the Apr-Jul runoff pulse at Lake Powell has been sliding for decades.

In [ ]:
runoff = runoff_vs_snow(settings.cache_dir, snow_year)

fig_scatter = build_snow_vs_runoff(runoff)
fig_scatter.show()

fig_trend = build_efficiency_trend(runoff)
fig_trend.show()

In [ ]:
latest_runoff = runoff.iloc[-1]
efficiency_trend = theil_sen_trend(runoff["water_year"], runoff["runoff_efficiency"])
display(Markdown(f"**{describe_runoff_year(latest_runoff)}**"))
display(Markdown(describe_trend(efficiency_trend, "Spring runoff efficiency")))

In [ ]:
# Robustness check (T16 carry-over note, decision 0025): a run of dry years can
# make runoff_efficiency look like it's declining on its own, since base flow
# doesn't shrink with snow. The trend in the *residuals* of a runoff-vs-peak-SWE
# regression isn't vulnerable to that, so we check it agrees in sign and
# significance before trusting the headline -- see decision 0025 for why the
# comparison is on tau/p rather than on pct_of_normal_per_decade.
robustness_trend = residual_efficiency_trend(runoff)
print(
    f"direct:  slope/decade={efficiency_trend.slope_per_decade:.3f}  "
    f"tau={efficiency_trend.kendall_tau:.2f}  p={efficiency_trend.p_value:.3f}"
)
print(
    f"resid.:  slope/decade={robustness_trend.slope_per_decade:.3f}  "
    f"tau={robustness_trend.kendall_tau:.2f}  p={robustness_trend.p_value:.3f}"
)
trends_agree = (
    efficiency_trend.slope_per_decade < 0 and robustness_trend.slope_per_decade < 0
)
assert trends_agree, (
    "the direct and residual trends disagree in sign -- revisit the chapter 3 narrative"
)

**How we know:** `runoff_efficiency` is Apr-Jul unregulated inflow at Lake
Powell (RISE item 512) in MAF, divided by that water year's peak basin SWE in
inches (decision 0017). The Theil-Sen trend above is robust to outliers, and
Kendall's tau tests whether the decline is more than noise. Because a run of
dry years alone can inflate this ratio's apparent decline (base flow doesn't
shrink with snow the way peak flow does), we also checked the trend in the
residuals of a runoff-vs-peak-SWE regression -- a check that isn't vulnerable to
that effect. Both trends are negative and both are statistically significant
(see the printed comparison above; decision 0025 has the numbers from the real
cache and explains why we compare the two trends' sign and Kendall's tau rather
than their `pct_of_normal_per_decade`, which isn't meaningful for a
near-zero-mean residual series). Source: USGS/RISE Powell unregulated inflow
and the SNOTEL index above.

## 4 · Earlier and faster

*(chapter 4 -- T21)*

## 5 · Dams flatten the river

*(chapter 5 -- T22)*

## 6 · The bank account

*(chapter 6 -- T23)*

## 2026 at a glance

*(KPI panel -- T25)*

## Sources and methods

*(T26)*